# Lab 8.3: Advanced RAG Techniques

**Course:** Advanced Natural Language Processing (NLP702/806)

**Instructor:** Dr. Fajri Koto

---

In this notebook, we explore techniques that improve upon the basic RAG pipeline:

1. **Chunking strategies** - How to split documents into passages
2. **Hybrid retrieval** - Combining BM25 with dense retrieval
3. **Re-ranking** - Using a cross-encoder to refine retrieval results
4. **Evaluation** - Measuring RAG quality systematically

In [11]:
import numpy as np
from collections import defaultdict

import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm import tqdm
from openai import OpenAI

## Section 1: Chunking Strategies

How you split documents into chunks dramatically affects retrieval quality. A chunk that is too long dilutes the signal; too short and it lacks context.

In [12]:
sample_document = """Albert Einstein was a German-born theoretical physicist who is widely held to be one of the greatest and most influential scientists of all time. He is best known for developing the theory of relativity, but he also made important contributions to quantum mechanics. His work is also known for its influence on the philosophy of science.

Einstein was born in the German Empire but moved to Switzerland in 1895. In 1905, he published four groundbreaking papers. These outlined the theory of the photoelectric effect, explained Brownian motion, introduced special relativity, and demonstrated mass-energy equivalence. Einstein received the Nobel Prize in Physics in 1921 for his explanation of the photoelectric effect.

After publishing his general theory of relativity in 1915, Einstein became world-famous. He was offered the presidency of Israel in 1952 but declined. Einstein died in Princeton, New Jersey in 1955."""

print(f"Document length: {len(sample_document)} characters, {len(sample_document.split())} words")

Document length: 918 characters, 141 words


In [13]:
def chunk_by_fixed_size(text, chunk_size=100, overlap=20):
    """Split text into fixed-size word chunks with overlap."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


def chunk_by_paragraph(text):
    """Split text by paragraph boundaries."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    return paragraphs


def chunk_by_sentence(text, sentences_per_chunk=3, overlap=1):
    """Split text into chunks of N sentences with overlap."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    start = 0
    while start < len(sentences):
        end = min(start + sentences_per_chunk, len(sentences))
        chunk = " ".join(sentences[start:end])
        chunks.append(chunk)
        start += sentences_per_chunk - overlap
    return chunks


# Compare chunking strategies
strategies = {
    "Fixed-size (50 words, 10 overlap)": chunk_by_fixed_size(sample_document, 50, 10),
    "Paragraph": chunk_by_paragraph(sample_document),
    "Sentence (3 per chunk, 1 overlap)": chunk_by_sentence(sample_document, 3, 1),
}

for name, chunks in strategies.items():
    print(f"\n{name} ({len(chunks)} chunks):")
    for i, chunk in enumerate(chunks):
        print(f"  [{i}] ({len(chunk.split())} words) {chunk[:100]}...")


Fixed-size (50 words, 10 overlap) (4 chunks):
  [0] (50 words) Albert Einstein was a German-born theoretical physicist who is widely held to be one of the greatest...
  [1] (50 words) to quantum mechanics. His work is also known for its influence on the philosophy of science. Einstei...
  [2] (50 words) of the photoelectric effect, explained Brownian motion, introduced special relativity, and demonstra...
  [3] (21 words) became world-famous. He was offered the presidency of Israel in 1952 but declined. Einstein died in ...

Paragraph (3 chunks):
  [0] (56 words) Albert Einstein was a German-born theoretical physicist who is widely held to be one of the greatest...
  [1] (54 words) Einstein was born in the German Empire but moved to Switzerland in 1895. In 1905, he published four ...
  [2] (31 words) After publishing his general theory of relativity in 1915, Einstein became world-famous. He was offe...

Sentence (3 per chunk, 1 overlap) (5 chunks):
  [0] (56 words) Albert Einstein was

**Chunking Guidelines:**

| Strategy | Best For | Pros | Cons |
|----------|----------|------|------|
| **Fixed-size** | Uniform text (e.g., articles) | Simple, predictable | May split mid-sentence |
| **Paragraph** | Well-structured documents | Preserves coherent units | Uneven chunk sizes |
| **Sentence** | QA, precise retrieval | Natural boundaries | Short chunks may lack context |
| **Recursive/semantic** | Mixed-format docs | Adapts to structure | More complex to implement |

**Rule of thumb:** 100-300 words per chunk with 10-20% overlap works well for most use cases.

## Section 2: Hybrid Retrieval (BM25 + Dense)

**Hybrid retrieval** combines the strengths of sparse (BM25) and dense retrieval using **Reciprocal Rank Fusion (RRF)**:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60).

In [14]:
# Load corpus
corpus_dataset = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")
passages = [item["passage"] for item in corpus_dataset]

# Use a manageable subset for dense retrieval
subset_size = min(5000, len(passages))
passages_subset = passages[:subset_size]

# Build BM25 index
tokenized = [p.lower().split() for p in tqdm(passages_subset, desc="Tokenizing for BM25")]
bm25_index = BM25Okapi(tokenized)

# Build dense index
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Encoding passages with dense model...")
passage_embeddings = dense_model.encode(passages_subset, show_progress_bar=True, batch_size=64)

print(f"\nIndexes built: BM25 + Dense over {subset_size} passages")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14912.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding passages with dense model...


Batches: 100%|██████████| 50/50 [00:01<00:00, 25.91it/s]


Indexes built: BM25 + Dense over 3200 passages


In [15]:
def bm25_search(query, top_k=20):
    """BM25 retrieval returning (index, score) pairs."""
    scores = bm25_index.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]


def dense_search(query, top_k=20):
    """Dense retrieval returning (index, score) pairs."""
    q_emb = dense_model.encode([query])
    sims = np.dot(passage_embeddings, q_emb.T).flatten()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(int(i), float(sims[i])) for i in top_idx]


def reciprocal_rank_fusion(rankings, k=60):
    """Fuse multiple ranked lists using RRF."""
    rrf_scores = defaultdict(float)
    
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking, 1):
            rrf_scores[doc_id] += 1.0 / (k + rank)
    
    # Sort by fused score
    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return fused


def hybrid_search(query, top_k=5):
    """Hybrid retrieval using BM25 + Dense with RRF."""
    bm25_results = bm25_search(query, top_k=20)
    dense_results = dense_search(query, top_k=20)
    
    fused = reciprocal_rank_fusion([bm25_results, dense_results])
    return fused[:top_k]

In [16]:
# Compare all three retrieval methods
test_queries = [
    "What is photosynthesis?",                      # Direct term match
    "How do plants convert sunlight into food?",     # Semantic paraphrase
    "largest ocean on Earth",                        # Short factoid
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("=" * 70)
    
    for method_name, search_fn in [("BM25", bm25_search), ("Dense", dense_search), ("Hybrid", hybrid_search)]:
        results = search_fn(query, top_k=3)
        print(f"\n  {method_name}:")
        for rank, (idx, score) in enumerate(results, 1):
            print(f"    [{rank}] (score={score:.4f}) {passages_subset[idx][:100]}...")


Query: 'What is photosynthesis?'

  BM25:
    [1] (score=6.8308) What seems clear is that penguins belong to a clade of Neoaves (living birds except paleognaths and ...
    [2] (score=5.7479) The expression is part of a conceptual framework for testing (see Duck test) of some computer system...
    [3] (score=5.6039) The expression "quacks like a duck" is sometimes a short form for "It looks like a duck, it quacks l...

  Dense:
    [1] (score=0.3288) Briefly, it can be explained thus:...
    [2] (score=0.3120) *By pulling down trees to eat leaves, breaking branches, and pulling out roots they create clearings...
    [3] (score=0.2745) Genus Lutrogale ...

  Hybrid:
    [1] (score=0.0164) What seems clear is that penguins belong to a clade of Neoaves (living birds except paleognaths and ...
    [2] (score=0.0164) Briefly, it can be explained thus:...
    [3] (score=0.0161) The expression is part of a conceptual framework for testing (see Duck test) of some computer system...

Query: '

## Section 3: Re-ranking with Cross-Encoders

A **cross-encoder** takes `(query, passage)` as a joint input and scores their relevance. It is more accurate than bi-encoder retrieval but too slow for first-stage retrieval over the full corpus. The standard pattern:

1. **First stage**: BM25 or bi-encoder retrieves top-20 candidates (fast)
2. **Second stage**: Cross-encoder re-ranks those 20 candidates (accurate)

In [17]:
# Load a cross-encoder reranker
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


def retrieve_and_rerank(query, top_k_retrieve=20, top_k_final=5):
    """Two-stage retrieval: BM25 first stage + cross-encoder reranking."""
    
    # Stage 1: BM25 retrieves candidates
    bm25_results = bm25_search(query, top_k=top_k_retrieve)
    
    # Stage 2: Cross-encoder reranks
    candidate_passages = [(query, passages_subset[idx]) for idx, _ in bm25_results]
    rerank_scores = reranker.predict(candidate_passages)
    
    # Combine and sort by reranker score
    reranked = []
    for i, (idx, bm25_score) in enumerate(bm25_results):
        reranked.append((idx, float(rerank_scores[i]), bm25_score))
    
    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked[:top_k_final]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7956.39it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
# Compare BM25-only vs BM25 + Reranker
query = "Who invented the light bulb?"

print(f"Query: '{query}'\n")

# BM25 only
bm25_results = bm25_search(query, top_k=5)
print("BM25 Only:")
for rank, (idx, score) in enumerate(bm25_results, 1):
    print(f"  [{rank}] (BM25={score:.3f}) {passages_subset[idx][:120]}...")

# BM25 + Reranker
reranked = retrieve_and_rerank(query, top_k_retrieve=20, top_k_final=5)
print(f"\nBM25 + Reranker:")
for rank, (idx, rerank_score, bm25_score) in enumerate(reranked, 1):
    print(f"  [{rank}] (rerank={rerank_score:.3f}, BM25={bm25_score:.3f}) {passages_subset[idx][:120]}...")

Query: 'Who invented the light bulb?'

BM25 Only:
  [1] (BM25=14.757) * Lomas, Robert, " The Man who Invented the Twentieth Century". Lecture to South Western Branch of Instititute of Physic...
  [2] (BM25=13.994) In mechanics, Newton enunciated the principles of conservation of momentum and angular momentum. In optics, he invented ...
  [3] (BM25=10.823) * Light passed through the so-called "vacuum" in the glass tube....
  [4] (BM25=10.279) Newton argued that light is composed of particles or corpuscles and were refracted by accelerating toward the denser med...
  [5] (BM25=10.141) From this work he concluded that any refracting telescope would suffer from the dispersion of light into colours, and in...

BM25 + Reranker:
  [1] (rerank=-1.985, BM25=13.994) In mechanics, Newton enunciated the principles of conservation of momentum and angular momentum. In optics, he invented ...
  [2] (rerank=-1.992, BM25=8.384) Nikola Tesla (Serbian Cyrillic: ÐÐ¸ÐºÐ¾Ð»Ð° Ð¢ÐµÑÐ»Ð°) (July 10 1856   7 

## Section 4: Evaluating RAG Systems

RAG evaluation has two parts:
1. **Retrieval quality** - Are we finding the right passages?
2. **Generation quality** - Is the final answer correct?

### 4.1 Retrieval Metrics

In [19]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    """What fraction of relevant docs are in the top-k retrieved?"""
    retrieved_set = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)
    if not relevant_set:
        return 0.0
    return len(retrieved_set & relevant_set) / len(relevant_set)


def precision_at_k(retrieved_ids, relevant_ids, k):
    """What fraction of top-k retrieved docs are relevant?"""
    retrieved_set = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)
    if not retrieved_set:
        return 0.0
    return len(retrieved_set & relevant_set) / len(retrieved_set)


def mrr(retrieved_ids, relevant_ids):
    """Mean Reciprocal Rank: 1/rank of the first relevant result."""
    relevant_set = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0


# Demonstration with a synthetic example
retrieved = [3, 7, 1, 5, 9, 2, 8, 4, 6, 0]
relevant = [1, 5, 8]

print(f"Retrieved order: {retrieved}")
print(f"Relevant docs:   {relevant}")
print()
for k in [1, 3, 5, 10]:
    print(f"  Recall@{k}: {recall_at_k(retrieved, relevant, k):.3f}  |  Precision@{k}: {precision_at_k(retrieved, relevant, k):.3f}")
print(f"  MRR: {mrr(retrieved, relevant):.3f}")

Retrieved order: [3, 7, 1, 5, 9, 2, 8, 4, 6, 0]
Relevant docs:   [1, 5, 8]

  Recall@1: 0.000  |  Precision@1: 0.000
  Recall@3: 0.333  |  Precision@3: 0.333
  Recall@5: 0.667  |  Precision@5: 0.400
  Recall@10: 1.000  |  Precision@10: 0.300
  MRR: 0.333


### 4.2 Generation Metrics

For evaluating the generated answers, we can use:
- **Exact Match (EM)**: Does the answer exactly match the gold answer?
- **F1 Score**: Token-level overlap between predicted and gold answer
- **ROUGE-L**: Longest common subsequence overlap

In [20]:
import re

def normalize_answer(text):
    """Normalize answer for evaluation: lowercase, strip punctuation and extra whitespace."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = ' '.join(text.split())
    return text


def exact_match(prediction, gold):
    """Check if normalized prediction matches gold answer."""
    return normalize_answer(prediction) == normalize_answer(gold)


def token_f1(prediction, gold):
    """Compute token-level F1 between prediction and gold."""
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()
    
    common = set(pred_tokens) & set(gold_tokens)
    if not common:
        return 0.0
    
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    
    return 2 * precision * recall / (precision + recall)


# Demonstration
examples = [
    ("George Washington", "George Washington"),
    ("George Washington was the first president.", "George Washington"),
    ("The first president was Washington.", "George Washington"),
    ("Abraham Lincoln", "George Washington"),
]

print(f"{'Prediction':<50} {'Gold':<25} {'EM':>5} {'F1':>6}")
print("-" * 90)
for pred, gold in examples:
    em = exact_match(pred, gold)
    f1 = token_f1(pred, gold)
    print(f"{pred:<50} {gold:<25} {em!s:>5} {f1:>6.3f}")

Prediction                                         Gold                         EM     F1
------------------------------------------------------------------------------------------
George Washington                                  George Washington          True  1.000
George Washington was the first president.         George Washington         False  0.500
The first president was Washington.                George Washington         False  0.286
Abraham Lincoln                                    George Washington         False  0.000


### Summary: The RAG Improvement Stack

```
                Accuracy
                   ^
                   |     * Rerank + Hybrid + Good Chunks
  Cross-encoder    |
  Reranking        |   * Hybrid + Good Chunks
                   |
  Hybrid           | * BM25 + Good Chunks
  Retrieval        |
                   | * BM25 (baseline)
                   |
                   | * No retrieval (LLM only)
                   +-------------------------------> Complexity
```

Each technique adds complexity but generally improves quality. Start with BM25 as your baseline and add complexity only when needed.